# Train an image classifier for the Raspberry Pi

This notebook trains an image classifier from **folders of example images** and exports it as a
**LiteRT (TensorFlow Lite) model** that runs on a Raspberry Pi.

**Workflow**

1. Get the images: one folder per class. The folder names become the labels.
2. Train a small model using **transfer learning** (MobileNetV2, pretrained on ImageNet).
3. Check how well it works on images it has never seen.
4. Export `model_int8.tflite`, `model_float.tflite` and `labels.txt`, and download them.

**Before you start:** *Runtime → Change runtime type → T4 GPU* if one is available.
A GPU makes training a few times faster, but the CPU runtime also works.

Run the cells in order with **Shift + Enter**.

## 1. Check the runtime

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", bool(gpus), gpus)
if not gpus:
    print("No GPU: training will still work, it will just take longer.")

## 2. Get the images

The model learns from images arranged as **one folder per class**. The folder names
become the labels:

```
my_images/
    keyboard/     img001.jpg  img002.jpg ...
    mouse/        img001.jpg  img002.jpg ...
    background/   img001.jpg  img002.jpg ...
```

**Run only ONE of the two cells below:**

- **Option A** downloads example images of cats and dogs. Use this the first time.
- **Option B** uploads your own images, made with `capture_images.py` on the Pi.

### Option A: example images (cats and dogs)

Downloads a random sample of Microsoft's *cats vs dogs* dataset from Hugging Face:
1,000 training and 250 test images per class. Takes a minute or two.

In [ ]:
!pip -q install datasets
import os
from datasets import load_dataset

N_TRAIN, N_TEST = 1000, 250          # images per class
ROOT = "cats_vs_dogs"

if not os.path.isdir(ROOT):
    hf = load_dataset("microsoft/cats_vs_dogs", split="train").shuffle(seed=42)
    names = hf.features["labels"].names
    wanted = {"train": N_TRAIN, "test": N_TEST}
    saved = {(s, n): 0 for s in wanted for n in names}
    for i, ex in enumerate(hf):
        name = names[ex["labels"]]
        split = next((s for s in wanted if saved[(s, name)] < wanted[s]), None)
        if split is None:
            if all(saved[k] >= wanted[k[0]] for k in saved):
                break
            continue
        folder = os.path.join(ROOT, split, name)
        os.makedirs(folder, exist_ok=True)
        ex["image"].convert("RGB").save(os.path.join(folder, f"{i}.jpg"), quality=90)
        saved[(split, name)] += 1

DATA_DIR = f"{ROOT}/train"   # one sub-folder per class
TEST_DIR = f"{ROOT}/test"    # held-back test set

for d in sorted(os.listdir(DATA_DIR)):
    print(f"{d:>12}: {len(os.listdir(os.path.join(DATA_DIR, d)))} images")

### Option B: your own images

You need the **`my_images.zip`** file that `capture_images.py` creates on the Pi
when you press **q**. It is in the same folder as the script (e.g. `~/classifier/my_images.zip`).

1. Run the cell below. A **Choose Files** button appears underneath it.
2. Click **Choose Files**, find `my_images.zip` and click **Open**.
3. Wait for the upload to reach 100%. The cell then unpacks the zip and lists your classes.

If you are using Colab in the Pi's own web browser, the file is in your home folder
under `classifier`. If you are on another computer, copy the zip there first (USB stick,
email to yourself, or `scp`).

A zip of your own folders also works, as long as it contains **one folder per class**.
Run the cell again to replace the images with a new upload.

In [ ]:
import os, shutil, zipfile
from google.colab import files

IMAGE_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".gif")
UPLOAD_DIR = "uploaded_images"

def has_images(folder):
    return any(f.lower().endswith(IMAGE_EXT) for f in os.listdir(folder))

def find_class_root(top):
    """Find the folder whose sub-folders hold the images (one sub-folder per class)."""
    for dirpath, dirnames, _ in os.walk(top):
        dirnames.sort()
        if sum(has_images(os.path.join(dirpath, d)) for d in dirnames) >= 2:
            return dirpath
    return None

uploaded = files.upload()          # click "Choose Files" and pick my_images.zip
zips = [name for name in uploaded if name.lower().endswith(".zip")]
assert zips, "Please upload a .zip file (e.g. my_images.zip)."

shutil.rmtree(UPLOAD_DIR, ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(UPLOAD_DIR)
os.remove(zips[0])

# Remove hidden files and folders that Macs and Windows add to zips (__MACOSX, ._*, .DS_Store)
for dirpath, dirnames, filenames in os.walk(UPLOAD_DIR, topdown=True):
    for d in [d for d in dirnames if d.startswith((".", "__"))]:
        shutil.rmtree(os.path.join(dirpath, d))
        dirnames.remove(d)
    for f in filenames:
        if f.startswith(".") or f.lower() in ("thumbs.db", "desktop.ini"):
            os.remove(os.path.join(dirpath, f))

DATA_DIR = find_class_root(UPLOAD_DIR)
assert DATA_DIR, "Couldn't find class folders in the zip. It needs one folder of images per class."
TEST_DIR = None                    # 20% of your images are used for testing instead

# Remove folders that don't contain images, so they don't become empty classes
for d in os.listdir(DATA_DIR):
    p = os.path.join(DATA_DIR, d)
    if os.path.isdir(p) and not has_images(p):
        shutil.rmtree(p)

print("Classes found in", DATA_DIR)
for d in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, d)
    if os.path.isdir(p):
        n = sum(f.lower().endswith(IMAGE_EXT) for f in os.listdir(p))
        print(f"{d:>15}: {n} images" + ("   <- very few, aim for 100+" if n < 50 else ""))

## 3. Load the images

Every image is resized to 224 × 224 pixels, the size MobileNetV2 expects.
20% of the training folder is held aside as a **validation set** to check progress during training.

In [ ]:
IMG_SIZE = 224
BATCH = 32
SEED = 42

train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="both", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

test_ds = None
if TEST_DIR:
    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=False)
    assert test_ds.class_names == class_names, "Test folders must match training folders"

### Look at some examples

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

## 4. Augmentation

With a few hundred images per class, the model can simply memorise them.
**Augmentation** makes random flips, small rotations, zooms and brightness changes each
time an image is used, so the model has to learn the subject rather than the exact photo.
It is only applied to the training set.

In [ ]:
from tensorflow.keras import layers

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (train_ds
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = val_ds.cache().prefetch(AUTOTUNE)
if test_ds is not None:
    test_ds = test_ds.cache().prefetch(AUTOTUNE)

## 5. Build the model (transfer learning)

Training an image model from scratch needs a huge dataset. Instead we take **MobileNetV2**,
already trained on 1.4 million ImageNet photos, **freeze** it, and train only a small new
layer on top that maps its features to *our* classes.

The model takes ordinary **RGB pixels from 0 to 255**. The scaling MobileNetV2 needs
(to the range −1 to +1) is built into the model, so the Pi doesn't have to get it right.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)   # 0..255 -> -1..1
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="probs")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary(show_trainable=True)

## 6. Train

An **epoch** is one pass through all the training images. Training stops early if the
validation accuracy stops improving, and the best version is kept.

In [ ]:
EPOCHS = 10
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True)])

In [ ]:
h = history.history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(h["accuracy"], label="train"); plt.plot(h["val_accuracy"], label="validation")
plt.title("Accuracy"); plt.xlabel("epoch"); plt.legend()
plt.subplot(1, 2, 2); plt.plot(h["loss"], label="train"); plt.plot(h["val_loss"], label="validation")
plt.title("Loss"); plt.xlabel("epoch"); plt.legend()
plt.show()

**Reading the curves:** if training accuracy keeps rising while validation accuracy falls,
the model is **overfitting** (memorising the training images). More varied images or more
augmentation usually helps.

## 7. Test on images the model has never seen

The **confusion matrix** shows which classes get mistaken for which.
Numbers on the diagonal are correct predictions.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

eval_ds = test_ds if test_ds is not None else val_ds
y_true = np.concatenate([y.numpy() for _, y in eval_ds])
y_pred = np.argmax(model.predict(eval_ds, verbose=0), axis=1)
print(f"Keras model accuracy: {np.mean(y_true == y_pred):.3f}")
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot(cmap="Blues")
plt.show()

## 8. Export for the Raspberry Pi

Two versions are exported so you can compare them:

| File | Weights | Size | Notes |
|---|---|---|---|
| `model_float.tflite` | 32-bit float | ~9 MB | Same accuracy as the Keras model |
| `model_int8.tflite` | 8-bit integer | ~3 MB | Faster on the Pi, usually tiny accuracy loss |

The int8 version is **quantized**: it needs a few hundred example images (the
*representative dataset*) to work out the range of values inside the network.

In [ ]:
import pathlib
OUT = pathlib.Path("export"); OUT.mkdir(exist_ok=True)
model.export(str(OUT / "saved_model"), verbose=False)

# Float model
conv = tf.lite.TFLiteConverter.from_saved_model(str(OUT / "saved_model"))
(OUT / "model_float.tflite").write_bytes(conv.convert())

# Full-integer (int8) model with uint8 input and output
raw_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, image_size=(IMG_SIZE, IMG_SIZE), batch_size=1, shuffle=True, seed=SEED)
def representative_data():
    for images, _ in raw_ds.take(300):
        yield [tf.cast(images, tf.float32)]

conv = tf.lite.TFLiteConverter.from_saved_model(str(OUT / "saved_model"))
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.uint8
conv.inference_output_type = tf.uint8
(OUT / "model_int8.tflite").write_bytes(conv.convert())

(OUT / "labels.txt").write_text("\n".join(class_names) + "\n")

for f in sorted(OUT.glob("*.tflite")):
    print(f"{f.name:>20}: {f.stat().st_size / 1e6:.1f} MB")
print("labels:", class_names)

## 9. Check the exported models

This runs the `.tflite` files with **LiteRT**, the same runtime used on the Pi,
so you can see whether quantization cost any accuracy before you download anything.

In [ ]:
!pip -q install ai-edge-litert
from ai_edge_litert.interpreter import Interpreter

def tflite_predict(path, images):
    it = Interpreter(model_path=str(path))
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    preds = []
    for img in images:
        x = img[np.newaxis].astype(np.float32)
        if inp["dtype"] != np.float32:                       # quantized input
            scale, zero = inp["quantization"]
            x = np.clip(np.round(x / scale + zero), 0, 255)
        it.set_tensor(inp["index"], x.astype(inp["dtype"]))
        it.invoke()
        preds.append(np.argmax(it.get_tensor(out["index"])[0]))
    return np.array(preds)

images = np.concatenate([x.numpy() for x, _ in eval_ds])
for name in ["model_float.tflite", "model_int8.tflite"]:
    acc = np.mean(tflite_predict(OUT / name, images) == y_true)
    print(f"{name:>20}: accuracy {acc:.3f}")

## 10. Download

This downloads `pi_model.zip` containing both models and `labels.txt`.

- **Using Colab in the Pi's browser:** the zip goes to `~/Downloads`. In a terminal:
  `cd ~/classifier && unzip -o ~/Downloads/pi_model.zip`
- **Using another computer:** copy the zip to `~/classifier` on the Pi (USB stick or `scp`),
  then `cd ~/classifier && unzip -o pi_model.zip`

`-o` replaces any older model files without asking.

In [ ]:
!cd export && zip -q -j ../pi_model.zip model_float.tflite model_int8.tflite labels.txt
from google.colab import files
files.download("pi_model.zip")

## Things to try

- Point the cat/dog model at something that is neither. It still answers "cat" or "dog",
  because it only knows those two classes. Then train your own model with a
  **background** class and compare.
- Collect your own images with `capture_images.py` and use Option B (section 2).
  Any number of classes works, and nothing else in the notebook needs to change.
- Compare the float and int8 models on the Pi: speed, accuracy, file size.
- Change `EPOCHS`, or remove the augmentation, and see what happens to the curves.